# CV16 – HAN Cap_01: Preprocessing + OCR (2 pages)

Notebook giữ nguyên thuật toán preprocessing, tách cột và PaddleOCR từ notebook gốc. Các thay đổi:
- Chỉ xử lý 2 trang đầu của `Cap_01/HAN.pdf`.
- Điều chỉnh đường dẫn theo cấu trúc thư mục mới.
- Lưu ảnh cột đã preprocessing vào `images/<page>/`.
- OCR từng ảnh cột và lưu kết quả vào `raw/<page>/`.
- Có cell riêng để xóa nội dung cũ trong `02_ocr_tho/Cap_01/HAN/raw/` trước khi chạy lại.

**Lưu ý:** Chạy cell xóa dữ liệu chỉ khi muốn làm sạch kết quả OCR cũ. Cell này chỉ tác động đến thư mục `raw` nói trên, không xóa PDF nguồn hoặc ảnh trang/ảnh cột.


## 1. Khai báo đường dẫn và phạm vi

In [ ]:
from pathlib import Path
import os

# Kaggle input: thư mục chứa Cap_01/HAN.pdf
data_path = Path("/kaggle/input/datasets/anhtuanxxx/nlp-midterm-phu-bien-tap-luc/01_nguon")

# Kaggle working
Project_dir = Path("/kaggle/working/CV16_nlp_midterm")

CAP = "Cap_01"
LANG = "HAN"
PDF_PATH = data_path / CAP / f"{LANG}.pdf"

# Chỉ chạy 2 trang đầu
PAGES_TO_PROCESS = [1, 2]

# Cấu trúc output mới
HAN_DIR = Project_dir / "02_ocr_tho" / CAP / LANG
IMAGES_DIR = HAN_DIR / "images"
RAW_DIR = HAN_DIR / "raw"

print("PDF:", PDF_PATH)
print("Project:", Project_dir)
print("PDF exists:", PDF_PATH.exists())


## 2. Tạo cấu trúc thư mục

In [ ]:
folders = [
    Project_dir / "01_nguon" / CAP,
    IMAGES_DIR,
    RAW_DIR,
    Project_dir / "02_ocr_tho" / CAP / LANG / "corrected",
    Project_dir / "03_text_sach",
    Project_dir / "04_dong_hang",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Đã tạo/kiểm tra cấu trúc thư mục.")


# 3. Xóa RAW folder - Hỏi trước khi chạy

In [ ]:
# CELL DESTRUCTIVE:
# Xóa toàn bộ nội dung bên trong 02_ocr_tho/Cap_01/HAN/raw/
# Không xóa chính thư mục raw, ảnh nguồn hoặc ảnh preprocessing.

import shutil

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục sẽ được làm sạch:", RAW_DIR)
confirm_delete = input("Nhập DELETE_RAW để xác nhận xóa toàn bộ nội dung raw: ")

if confirm_delete == "DELETE_RAW":
    for item in RAW_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()
    print("Đã xóa nội dung cũ trong raw/.")
else:
    print("Đã bỏ qua thao tác xóa.")


## 3. PDF → ảnh trang (2 trang đầu)

In [ ]:
%pip install -q pymupdf


In [ ]:
import pymupdf
from tqdm.auto import tqdm

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy PDF: {PDF_PATH}")

with pymupdf.open(PDF_PATH) as doc:
    total_pages = doc.page_count
    print("Tổng số trang PDF:", total_pages)

    for page_no in PAGES_TO_PROCESS:
        if page_no < 1 or page_no > total_pages:
            raise ValueError(f"Trang {page_no} không hợp lệ; PDF có {total_pages} trang.")

        page = doc[page_no - 1]
        pix = page.get_pixmap(dpi=144)

        page_name = f"{page_no:04d}"
        page_image_dir = IMAGES_DIR / page_name
        page_image_dir.mkdir(parents=True, exist_ok=True)

        image_path = page_image_dir / f"{page_name}.png"
        pix.save(image_path)
        print("Đã lưu:", image_path)


## 5. Kiểm tra ảnh trang

In [ ]:
from PIL import Image
from IPython.display import display

for page_no in PAGES_TO_PROCESS:
    page_name = f"{page_no:04d}"
    image_path = IMAGES_DIR / page_name / f"{page_name}.png"
    print(page_name, "exists:", image_path.exists())
    if image_path.exists():
        img = Image.open(image_path)
        print("Kích thước:", img.size, "| Mode:", img.mode)
        display(img)


## 6. Preprocessing và tách cột

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

def preprocess_page(image_path, max_side=2000):
    img = cv2.imread(str(image_path))
    if img is None:
        return None

    h, w = img.shape[:2]
    img = img[0:int(h * 0.92), :]

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(
        gray, None, h=10, templateWindowSize=7, searchWindowSize=21
    )
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    h_res, w_res = gray.shape
    s = max_side / max(h_res, w_res)
    if s != 1:
        gray = cv2.resize(
            gray,
            (int(w_res * s), int(h_res * s)),
            interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
        )
    return gray


def extract_columns_final(gray_img):
    _, thresh = cv2.threshold(
        gray_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Giữ nguyên logic tách cột từ notebook gốc
    vertical_projection = np.sum(thresh, axis=0)
    smoothed_proj = median_filter(vertical_projection, size=9)

    threshold = np.max(smoothed_proj) * 0.10
    is_gap = smoothed_proj < threshold

    h, w = gray_img.shape
    mid_start = int(w * 0.40)
    mid_end = int(w * 0.60)
    if mid_end > mid_start:
        spine_x = mid_start + np.argmax(smoothed_proj[mid_start:mid_end])
        gap_radius = 20
        is_gap[max(0, spine_x - gap_radius):min(w, spine_x + gap_radius)] = True

    columns = []
    in_column = False
    start_x = 0

    for x in range(len(is_gap)):
        if not is_gap[x] and not in_column:
            start_x = x
            in_column = True
        elif is_gap[x] and in_column:
            end_x = x
            in_column = False
            width = end_x - start_x

            if width > 20:
                if width > 150:
                    mid_c = start_x + width // 2
                    search_range = 40
                    s_start = max(start_x, mid_c - search_range)
                    s_end = min(end_x, mid_c + search_range)

                    if s_end > s_start:
                        split_x = s_start + np.argmin(smoothed_proj[s_start:s_end])
                        col1 = gray_img[:, max(0, start_x-10):min(w, split_x+10)]
                        col2 = gray_img[:, max(0, split_x-10):min(w, end_x+10)]
                        columns.extend([col1, col2])
                        continue

                pad = 15
                col_img = gray_img[:, max(0, start_x-pad):min(w, end_x+pad)]
                columns.append(col_img)

    if in_column:
        col_img = gray_img[:, max(0, start_x-15):]
        if col_img.shape[1] > 20:
            columns.append(col_img)

    return columns[::-1]


## 7. Chạy preprocessing và lưu ảnh cột

In [ ]:
PREPROCESSED = {}

for page_no in PAGES_TO_PROCESS:
    page_name = f"{page_no:04d}"
    page_image_path = IMAGES_DIR / page_name / f"{page_name}.png"

    gray_processed = preprocess_page(page_image_path)
    if gray_processed is None:
        raise RuntimeError(f"Không đọc được ảnh: {page_image_path}")

    cols = extract_columns_final(gray_processed)
    PREPROCESSED[page_name] = cols

    page_dir = IMAGES_DIR / page_name
    for i, col_img in enumerate(cols, start=1):
        col_name = f"{page_name}-{i:02d}.png"
        col_path = page_dir / col_name
        cv2.imwrite(str(col_path), col_img)

    print(f"Trang {page_name}: tìm thấy {len(cols)} cột.")

    cols_per_row = 6
    rows = max(1, (len(cols) + cols_per_row - 1) // cols_per_row)
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for i, col_img in enumerate(cols):
        axes[i].imshow(col_img, cmap="gray")
        axes[i].set_title(f"Cột {i+1}", fontsize=10)
        axes[i].axis("off")

    for i in range(len(cols), len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


## 8. Cài đặt PaddleOCR

In [ ]:
%pip uninstall -y torch torchvision torchaudio


In [ ]:
import subprocess, sys, glob, shutil, importlib.util

site = "/usr/local/lib/python3.13/dist-packages"

subprocess.call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch", "torchvision", "torchaudio"
])

for pattern in [
    "torch", "torch-*", "torchvision", "torchvision-*",
    "torchaudio", "torchaudio-*", "functorch", "torchgen"
]:
    for p in glob.glob(f"{site}/{pattern}"):
        print("Removing", p)
        shutil.rmtree(p, ignore_errors=True)

print("torch still found:", importlib.util.find_spec("torch") is not None)


In [ ]:
import os, sys, subprocess, importlib, importlib.util, warnings
warnings.filterwarnings("ignore", message="No ccache found")

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

installed_something = False

if importlib.util.find_spec("paddle") is None:
    print("Installing paddlepaddle-gpu...")
    pip_install(
        "paddlepaddle-gpu==3.2.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
    )
    installed_something = True

if importlib.util.find_spec("paddleocr") is None:
    print("Installing paddleocr...")
    pip_install("paddleocr")
    installed_something = True

if installed_something:
    print("Cài đặt xong. Hãy restart kernel, sau đó chạy lại notebook từ đầu.")
else:
    print("Packages: OK (already installed)")


## 9. Kiểm tra GPU và khởi tạo OCR

In [ ]:
import paddle
from paddleocr import PaddleOCR
from importlib.metadata import version

print("PaddleOCR version:", version("paddleocr"))
n_gpu = paddle.device.cuda.device_count()
print("Paddle version:", paddle.__version__)
print("GPUs khả dụng:", n_gpu)

if n_gpu == 0:
    raise RuntimeError("Không tìm thấy GPU. Hãy bật GPU Accelerator trong Kaggle.")

paddle.set_device("gpu:0")
conv = paddle.nn.Conv2D(3, 16, 3)
out_test = conv(paddle.randn([1, 3, 640, 640]))
print("Test Conv2D trên GPU 0 thành công:", out_test.shape)


In [ ]:
# Giữ nguyên cấu hình PaddleOCR từ notebook gốc
common_han = dict(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
)

ocr_han = PaddleOCR(
    lang="chinese_cht",
    device="gpu:0",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_det_limit_side_len=1600,
    text_det_limit_type="max",
    text_det_box_thresh=0.3,
    text_det_thresh=0.2,
    text_det_unclip_ratio=2.0,
    text_rec_score_thresh=0.0,
)

print("Đã khởi tạo ocr_han (chinese_cht) trên GPU 0.")


## 10. OCR các cột và lưu result

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR.mkdir(parents=True, exist_ok=True)

for page_no in PAGES_TO_PROCESS:
    page_name = f"{page_no:04d}"
    page_image_dir = IMAGES_DIR / page_name
    out_page_dir = RAW_DIR / page_name
    out_page_dir.mkdir(parents=True, exist_ok=True)

    col_files = sorted(page_image_dir.glob(f"{page_name}-*.png"))

    print(f"\n{'='*40}")
    print(f"ĐANG XỬ LÝ TRANG: {page_name}")
    print(f"Số cột: {len(col_files)}")
    print(f"{'='*40}")

    page_full_text = []
    page_avg_scores = []
    log_details = []

    for i, col_path in enumerate(col_files):
        col_name = col_path.stem

        # Chỉ tạo một thư mục cho mỗi cột
        col_out_dir = out_page_dir / col_name
        col_out_dir.mkdir(parents=True, exist_ok=True)

        # Đường dẫn file đầu ra cụ thể
        json_path = col_out_dir / f"{col_name}_res.json"
        img_path = col_out_dir / f"{col_name}_ocr_res_img.png"
        txt_path = col_out_dir / "result.txt"

        result = ocr_han.predict(
            str(col_path),
            text_det_limit_side_len=1600,
            text_det_limit_type="max",
            text_det_box_thresh=0.3,
            text_det_unclip_ratio=1.5
        )

        col_text = ""
        col_scores = []

        img_with_boxes = cv2.imread(str(col_path))

        if img_with_boxes is None:
            continue

        for res in result:
            # Lưu JSON và ảnh trực tiếp vào thư mục cột
            res.save_to_json(str(json_path))
            res.save_to_img(str(img_path))

            # Trích xuất văn bản và confidence
            if (
                "dt_polys" in res
                and "rec_texts" in res
                and "rec_scores" in res
            ):
                items = list(zip(
                    res["dt_polys"],
                    res["rec_texts"],
                    res["rec_scores"]
                ))

                # Sắp xếp các dòng từ trên xuống dưới
                items.sort(key=lambda x: min(pt[1] for pt in x[0]))

                for poly, text, score in items:
                    col_text += text
                    col_scores.append(score)

                    pts = np.array(poly, np.int32).reshape((-1, 1, 2))
                    cv2.polylines(
                        img_with_boxes,
                        [pts],
                        isClosed=True,
                        color=(0, 255, 0),
                        thickness=2
                    )

        # Confidence trung bình của cột
        avg_score = np.mean(col_scores) if col_scores else 0.0

        if col_scores:
            page_avg_scores.append(avg_score)

        log_details.append(
            f"[{col_name}] (Score: {avg_score*100:.1f}%): {col_text}"
        )

        # Lưu TXT của từng cột
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(col_text)

        page_full_text.append(col_text)

        if i < 3:
            plt.figure(figsize=(4, 10))
            plt.imshow(cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB))
            plt.title(f"{col_name} - Score: {avg_score*100:.1f}%")
            plt.axis("off")
            plt.show()

        print(f"Đã lưu: {col_name}")

    # Tổng hợp kết quả OCR theo trang
    final_page_text = "".join(page_full_text)
    page_score = np.mean(page_avg_scores) if page_avg_scores else 0.0

    txt_out_path = out_page_dir / f"{page_name}_reconstructed.txt"

    with open(txt_out_path, "w", encoding="utf-8") as f:
        f.write(f"=== TRANG: {page_name} ===\n")
        f.write(f"Độ tin cậy trung bình trang: {page_score*100:.2f}%\n")
        f.write("-" * 50 + "\n\n")
        f.write("--- CHI TIẾT TỪNG CỘT (TỪ PHẢI SANG TRÁI) ---\n")
        f.write("\n".join(log_details) + "\n\n")
        f.write("-" * 50 + "\n")
        f.write("--- NỘI DUNG GHÉP LIỀN MẠCH ---\n")
        f.write(final_page_text)

    print(f"Hoàn thành trang {page_name}: {out_page_dir}")

print("HOÀN TẤT OCR.")

## 11. Kiểm tra cây output

Kết quả chính:
- `images/0001/0001.png`: ảnh trang 1.
- `images/0001/0001-01.png`, `0001-02.png`, ...: ảnh cột sau preprocessing.
- `raw/0001/`: JSON OCR theo cột, ảnh kết quả có bounding box và file text ghép trang.
- Tương tự với trang `0002`.


In [ ]:
def print_tree(root, prefix=""):
    root = Path(root)
    for path in sorted(root.iterdir()):
        print(prefix + path.name + ("/" if path.is_dir() else ""))
        if path.is_dir():
            print_tree(path, prefix + "    ")

print_tree(HAN_DIR)


In [ ]:
import shutil
from IPython.display import FileLink, display

# Nén thư mục raw
zip_path = shutil.make_archive(
    str(HAN_DIR / "raw_Cap01_HAN"),
    "zip",
    root_dir=HAN_DIR
)

print(f"Đã tạo file: {zip_path}")

# Hiển thị link tải xuống
display(FileLink(zip_path))